<a href="https://colab.research.google.com/github/office268/A-ipynb/blob/main/ex_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- שלב 0: התקנת ספריות (אם עדיין לא הותקנו) ---
!pip install -q -U langchain langchain-community langchain-google-genai chromadb pypdf

import os
from google.colab import files, userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

# API הגדרת מפתח ה
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# 2. העלאת קובץ מהמכשיר
print("אנא העלה קובץ PDF מהמכשיר שלך:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

In [ ]:
# 3. טעינת הקובץ ופירוק לצ'אנקים של 500
print(f"\nמעבד את הקובץ: {file_name}...")
loader = PyPDFLoader(file_name)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
texts = text_splitter.split_documents(documents)
print(f"הקובץ פוצל ל-{len(texts)} צ'אנקים.")

In [ ]:
# 4. יצירת ה-Vector Store (בסיס נתונים וקטורי)
print("יוצר Embeddings ושומר בבסיס הנתונים הוקטורי...")
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

In [ ]:
# 5. הגדרת המודל ושרשרת השאלות (QA Chain)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4})
)

In [ ]:
# 6. הרצת שאילתה לבדיקה
print("\n" + "="*30)
query = input("מה תרצה לשאול על הקובץ שהעלית? ")
if query:
    print("\nחושב על תשובה...")
    response = qa_chain.invoke(query)
    print("\n--- תשובה מה-RAG ---")
    print(response["result"])